# Week 8 Notebook: VoltCore Energy — Battery Production (Parts 1 and 2)

*OMAIB · Module 1 · Week 8*

*© H. Sami Karaca. Questrom School of Business, Boston University.
Online Master of Science in Artificial Intelligence in Business.
Licensed for use in this course only; not for redistribution.*

**What this notebook is.** This is a practice notebook: it is not submitted and does not affect your grade. It verifies the two production plans you worked out on the case page, by encoding the same problem as a linear program and solving it directly. The notebook works like a calculator, not an oracle: it takes the same three components you met this week — decision variables, an objective function, and constraints — and returns the arithmetic answer. Confirming the inputs are right and interpreting the output is your job.

**How to read every output.** Check three things each time you run a solve cell: the **solution status** (the solver confirms it found an optimal solution), the **variable values** (the recommended Standard-Range and Long-Range quantities), and the **objective value** (total revenue at that mix).

---

## The Case (Part 1)

VoltCore's plant in Coweta County, Georgia, builds two battery packs this quarter: the **Standard-Range (SR)** pack, the higher-volume product, and the **Long-Range (LR)** pack, built on a pricier silicon-anode chemistry. Both move through shared production lines, each with a fixed number of hours available this quarter. Demand is firm for as many packs of each type as the plant can make.

| Production line | SR hours per pack | LR hours per pack | Hours available |
| --- | --- | --- | --- |
| Silicon-anode line | not used | 0.2 | 5,000 |
| Electrode coating line | 0.5 | 0.6 | 17,000 |
| Pack assembly line | 0.5 | 0.2 | 12,000 |

Wholesale prices: SR **\$4,000** per pack, LR **\$6,000** per pack. **The plant manager's goal: maximize this quarter's revenue.**

# Environment Setup

Before running any cells, select **File › Save a copy in Drive** so your work saves to your own Google Drive. This notebook loads everything it needs automatically — there's nothing for you to upload. Run the cells in order. This version uses the libraries already in your session. Results may vary with the runtime; use your own output in your answers.

In [ ]:
# Install the Pyomo optimization modeling package
!pip install -q pyomo

# Install the optimization engine (solution algorithms)
# GLPK linear-program solver: https://www.gnu.org/software/glpk/
!apt-get install -y -qq glpk-utils

import pyomo.environ as pyo
from pyomo.opt import SolverFactory

## Part 1 — The Georgia Plant: Maximize Revenue

Run the cell below without changing anything, then read the output top to bottom: status, plan, revenue.

In [ ]:
# ------------------------------------
# Prices (this quarter)               |
# ------------------------------------
SR_PRICE = 4000   # Standard-Range wholesale price, $ per pack
LR_PRICE = 6000   # Long-Range wholesale price, $ per pack

# Initialize the model by defining a pyomo instance
model = pyo.ConcreteModel()

# ------------------------------------
# Decision Variables                  |
# ------------------------------------
# SR = Standard-Range packs to build this quarter
# LR = Long-Range packs to build this quarter
model.SR = pyo.Var(bounds=(0, None))
model.LR = pyo.Var(bounds=(0, None))

# Shortcut: redefine pyomo variable names to use shorthand
SR = model.SR
LR = model.LR

# ------------------------------------
# Objective Function                  |
# ------------------------------------
# Maximize this quarter's revenue
model.obj = pyo.Objective(expr = SR_PRICE*SR + LR_PRICE*LR, sense = pyo.maximize)

# ------------------------------------
# Constraints                         |
# ------------------------------------
# One constraint per production line (hours used <= hours available)
model.SiliconAnode = pyo.Constraint(expr = 0.0*SR + 0.2*LR <= 5000,  doc='Silicon-anode line hours')
model.Coating      = pyo.Constraint(expr = 0.5*SR + 0.6*LR <= 17000, doc='Electrode coating line hours')
model.Assembly     = pyo.Constraint(expr = 0.5*SR + 0.2*LR <= 12000, doc='Pack assembly line hours')

# ------------------------------------
# Solve the Model                     |
# ------------------------------------
# GLPK solves linear programs; 'ranges' also writes a sensitivity report
# (reading that report is next week's skill -- a sneak peek waits at the end)
opt = SolverFactory('glpk')
opt.options['ranges'] = 'sens_part1.txt'
results = opt.solve(model, symbolic_solver_labels=True)

# ------------------------------------
# Print Results                       |
# ------------------------------------
print('--------------- RESULTS: PART 1 --------------- \n')
print('Solution status      :', results.solver.termination_condition)
print('Standard-Range packs :', f'{round(SR()):,}')
print('Long-Range packs     :', f'{round(LR()):,}')
print('Quarterly revenue    :', f'${round(model.obj()):,}')

In [ ]:
# How much of each line does the optimal plan use?
!pip install -q tabulate
from tabulate import tabulate as tb

rows = [['Silicon-anode',     0.0*SR() + 0.2*LR(), 5000],
        ['Electrode coating', 0.5*SR() + 0.6*LR(), 17000],
        ['Pack assembly',     0.5*SR() + 0.2*LR(), 12000]]

print(tb([[name, f'{used:,.0f}', f'{cap:,}'] for name, used, cap in rows],
         headers=['Production line', 'Hours used', 'Hours available'],
         tablefmt='fancy_grid'))

**Check your reading.** The status should be `optimal`, the plan **25,000 Long-Range and 4,000 Standard-Range packs**, and revenue **\$166,000,000** — the same plan Priya and Marcus reasoned to on the case page, confirmed to the dollar. Notice in the line-usage table which lines are used to the last hour (silicon-anode and coating) and which has room to spare (assembly).

If you asked a GenAI tool to solve Part 1 (Step 1 on the Colab page), put its answer next to these numbers — that comparison goes in your audit entry.

## Part 2 — One Number Changes

A competitor launches a comparable long-range pack, and VoltCore's sales team cuts the Long-Range price from \$6,000 to **\$4,500**. Same two packs, same three lines, same hours. The cell below is identical to Part 1 except one line: `LR_PRICE = 4500`.

Before you run it, ask the plant manager's question: does the Part 1 plan still hold?

In [ ]:
# ------------------------------------
# Prices (after the price cut)        |
# ------------------------------------
SR_PRICE = 4000   # unchanged
LR_PRICE = 4500   # was 6000 -- the only number that changed

# Initialize a fresh model
model = pyo.ConcreteModel()

# ------------------------------------
# Decision Variables                  |
# ------------------------------------
model.SR = pyo.Var(bounds=(0, None))
model.LR = pyo.Var(bounds=(0, None))
SR = model.SR
LR = model.LR

# ------------------------------------
# Objective Function                  |
# ------------------------------------
model.obj = pyo.Objective(expr = SR_PRICE*SR + LR_PRICE*LR, sense = pyo.maximize)

# ------------------------------------
# Constraints (identical to Part 1)   |
# ------------------------------------
model.SiliconAnode = pyo.Constraint(expr = 0.0*SR + 0.2*LR <= 5000,  doc='Silicon-anode line hours')
model.Coating      = pyo.Constraint(expr = 0.5*SR + 0.6*LR <= 17000, doc='Electrode coating line hours')
model.Assembly     = pyo.Constraint(expr = 0.5*SR + 0.2*LR <= 12000, doc='Pack assembly line hours')

# ------------------------------------
# Solve the Model                     |
# ------------------------------------
opt = SolverFactory('glpk')
opt.options['ranges'] = 'sens_part2.txt'
results = opt.solve(model, symbolic_solver_labels=True)

# ------------------------------------
# Print Results                       |
# ------------------------------------
print('--------------- RESULTS: PART 2 --------------- \n')
print('Solution status      :', results.solver.termination_condition)
print('Standard-Range packs :', f'{round(SR()):,}')
print('Long-Range packs     :', f'{round(LR()):,}')
print('Quarterly revenue    :', f'${round(model.obj()):,}')

# Line usage at the new optimal plan
from tabulate import tabulate as tb
rows = [['Silicon-anode',     0.0*SR() + 0.2*LR(), 5000],
        ['Electrode coating', 0.5*SR() + 0.6*LR(), 17000],
        ['Pack assembly',     0.5*SR() + 0.2*LR(), 12000]]
print()
print(tb([[name, f'{used:,.0f}', f'{cap:,}'] for name, used, cap in rows],
         headers=['Production line', 'Hours used', 'Hours available'],
         tablefmt='fancy_grid'))

**Check your reading.** The optimal plan is now **19,000 Standard-Range and 12,500 Long-Range packs**, for **\$132,250,000** — \$3,750,000 more than running the old plan at the new price (\$128,500,000). Long-Range still carries the higher sticker price, yet the model cut it in half. The case page explains why: the ranking that governs this plant is revenue per hour of the binding coating line, and the price cut flipped it. Notice the silicon-anode line in the table above — the hard ceiling of Part 1 now sits half used. Assembly is now also at capacity: the new plan sits where those two lines meet. Coating is still the one that decided the shift — it's the line whose per-hour ranking flipped.

If you asked your GenAI tool about Part 2, this is the verified answer to set beside it.

## Beat the Optimizer (optional)

Can you find a feasible plan that beats the model? Type any plan you like, pick a part, and run the cell. It checks your plan against the three line-hour constraints and tells you whether it is infeasible, feasible, or optimal.

In [ ]:
# Choose a part (1 = original prices, 2 = after the price cut), then type your plan
import pandas as pd

PART  = 1        # 1 or 2
my_SR = 12000    # your Standard-Range packs
my_LR = 8000     # your Long-Range packs

lr_price   = 6000 if PART == 1 else 4500
my_revenue = 4000*my_SR + lr_price*my_LR

# Does your plan respect each line's hours?
silicon_ok  = 0.0*my_SR + 0.2*my_LR <= 5000
coating_ok  = 0.5*my_SR + 0.6*my_LR <= 17000
assembly_ok = 0.5*my_SR + 0.2*my_LR <= 12000
feasible    = silicon_ok and coating_ok and assembly_ok

best = 166000000 if PART == 1 else 132250000
if not feasible:
    verdict = 'INFEASIBLE -- at least one line is over its available hours'
elif my_revenue > best:
    verdict = 'YOU BEAT THE OPTIMIZER -- a feasible plan above the verified best! Check your entries once more, then tell your professor.'
elif my_revenue == best:
    verdict = 'OPTIMAL -- this matches the verified best plan'
else:
    verdict = f'FEASIBLE, NOT OPTIMAL -- the best plan earns ${best - my_revenue:,.0f} more'

pd.DataFrame({'part': [PART], 'my_SR': [my_SR], 'my_LR': [my_LR],
              'my_revenue': [f'${my_revenue:,}'],
              'silicon_ok': [silicon_ok], 'coating_ok': [coating_ok], 'assembly_ok': [assembly_ok],
              'verdict': [verdict]})

## Sneak Peek: Sensitivity Analysis (part of Week 9)

Each solve also wrote a sensitivity report; here is Part 1's. Next week teaches you to read every line of a report like this — shadow prices, allowable ranges, and what they let a manager do. For now, just look at its shape: two tables, one about the constraints, one about the objective.

In [ ]:
# Display the sensitivity report GLPK wrote for Part 1
# (Part 2's is in sens_part2.txt in the file browser, if you're curious --
#  its numbers tell the story of the flip)
with open('sens_part1.txt', 'r') as f:
    print(f.read())

---
## Before You Leave

For your **VoltCore Audit Log** assignment on Blackboard, you now have everything you need: the verified plans for both parts, and your GenAI tool's answers to set beside them. Record how each pair compared and what you noticed about the tool's reasoning. The notebook itself is not submitted.

*Next week the case grows: a fuller order book, then a four-plant global network — and the sensitivity report above becomes the main event.*